<a href="https://colab.research.google.com/github/harshchill/AP-assignment/blob/main/sentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the first tme for me


In [ ]:
import pandas as pd

# Using engine='python' and on_bad_lines to handle the EOF/quoting issues
df = pd.read_csv('IMDB Dataset.csv', engine='python', on_bad_lines='skip')
print(df.head(5))
print(f'\nTotal rows loaded: {len(df)}')
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Total rows loaded: 50000
(50000, 2)


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# We need to download the list of English stopwords first
nltk.download('stopwords')

# Initialize our tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Strip out HTML tags using Regex
    text = re.sub(r'<.*?>', ' ', text)

    # 2. Remove all punctuation and numbers (keep only letters a-z)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # 3. Convert all text to lowercase
    text = text.lower()

    # 4. Tokenize (split the long string into an array of individual words)
    words = text.split()

    # 5. Remove stopwords AND apply stemming in one quick list comprehension
    # This is like running a filter() and map() over your array of words
    cleaned_words = [stemmer.stem(word) for word in words if word not in stop_words]

    # 6. Join the cleaned array of words back into a single string
    return ' '.join(cleaned_words)

# Test it on a single, messy string to see how it works!
sample_messy_review = "I LOVED this movie! <br /><br /> The acting was brilliantly done, 10/10."
print("Test Output:", clean_text(sample_messy_review))

Test Output: love movi act brilliantli done


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Create a new column in our "database" to hold the cleaned data
print("Cleaning 50,000 reviews... this might take a minute...")

df['cleaned_review'] = df['review'].apply(clean_text)

print("Cleanup complete!")

# Let's compare the before and after for the very first review
print("\n--- ORIGINAL ---")
print(df['review'].iloc[0])

print("\n--- CLEANED ---")
print(df['cleaned_review'].iloc[0])

Cleaning 50,000 reviews... this might take a minute...
Cleanup complete!

--- ORIGINAL ---
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreemen

In [ ]:
# Convert 'positive' to 1 and 'negative' to 0 using map
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print("Check out the first few labels now:")
print(df[['sentiment']].head(10))



Check out the first few labels now:
   sentiment
0          1
1          1
2          1
3          0
4          1
5          1
6          1
7          0
8          0
9          1


In [ ]:
from sklearn.model_selection import train_test_split

# X is our input (the cleaned reviews)
# y is our target (the 1s and 0s)
X = df['cleaned_review']
y = df['sentiment']

# We split: 80% for training, 20% for testing
# random_state is just a seed so your results stay consistent
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} reviews, testing on {len(X_test)} reviews.")

Training on 40000 reviews, testing on 10000 reviews.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize the vectorizer
# max_features=5000 means we only care about the top 5000 most important words
tfidf = TfidfVectorizer(max_features=5000)

# 2. Fit and transform the training data
# 'Fit' means learn the vocabulary; 'Transform' means turn it into numbers
X_train_tfidf = tfidf.fit_transform(X_train)

# 3. ONLY transform the test data (don't fit it, we want it to be a surprise!)
X_test_tfidf = tfidf.transform(X_test)

print("Words have been turned into a matrix of numbers!")
print("Matrix shape:", X_train_tfidf.shape)

Words have been turned into a matrix of numbers!
Matrix shape: (40000, 5000)


First model training

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize the model
nb_model = MultinomialNB()

# 2. Train the model (This is where the 'learning' happens)
nb_model.fit(X_train_tfidf, y_train)

# 3. Make predictions on the test data
y_pred = nb_model.predict(X_test_tfidf)

# 4. Check the results
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nDetailed Report:\n", classification_report(y_test, y_pred))

Naive Bayes Accuracy: 85.11%

Detailed Report:
               precision    recall  f1-score   support

           0       0.85      0.84      0.85      4961
           1       0.85      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

